<a href="https://colab.research.google.com/github/adiacla/Agentes/blob/main/diffusers/diffusers_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![diffusers_library](https://github.com/huggingface/diffusers/raw/main/docs/source/en/imgs/diffusers_library.jpg)

*Presentación de la nueva librería de Hugging Face para modelos de difusión*

Los modelos de difusión demostraron ser muy efectivos en la síntesis artificial, incluso superando a los GAN en imágenes. Debido a esto, ganaron tracción en la comunidad de aprendizaje automático y juegan un papel importante para sistemas como [DALL-E 2](https://openai.com/dall-e-2/) o [Imagen](https://imagen.research.google/) para generar imágenes fotorrealistas cuando se les da un texto.

Aunque los éxitos más prolíficos de los modelos de difusión han sido en la comunidad de visión por computadora, estos modelos también han logrado resultados notables en otros dominios, como:
- [generación de video](https://video-diffusion.github.io/),
- [síntesis de audio](https://diffwave-demo.github.io/),
- [aprendizaje por refuerzo](https://diffusion-planning.github.io/),
- y más.

Sin embargo, la mayor parte de la investigación reciente sobre modelos de difusión, *por ejemplo*, DALL-E 2 e Imagen, lamentablemente no es accesible para la comunidad de aprendizaje automático en general y normalmente permanece a puerta cerrada.

Aquí es donde entra la librería `diffusers` con los objetivos de:

1. reunir modelos de difusión recientes de repositorios independientes en un proyecto único y **mantenido a largo plazo** que sea construido por y para la **comunidad**,
2. reproducir sistemas de aprendizaje automático de alto impacto como DALLE e Imagen de una manera accesible para el público, y
3. crear una API fácil de usar que permita entrenar sus propios modelos o reutilizar puntos de control de otros repositorios para inferencia.

Este cuaderno le guiará a través de las características más importantes de `diffusers`.

Asumimos que el lector tiene una comprensión mínima de cómo funcionan los modelos de difusión. Para refrescar algo de teoría y terminología, recomendamos leer/escanear las siguientes publicaciones de blog:

  - Lilian Weng, OpenAI, [publicación introductoria](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)
  - Yang Song, Stanford, [publicación introductoria](https://yang-song.github.io/blog/2021/score/)
  - The Annotated Diffusion Model [publicación](https://huggingface.co/blog/annotated-diffusion)

O artículos:
- El artículo original que propone [termodinámica para el aprendizaje no supervisado](https://arxiv.org/abs/1503.03585),
- El artículo para un popular modelo de difusión, [Denoising Diffusion Probabilistic Models DDPM](https://arxiv.org/abs/2006.11239), o
- Un artículo reciente que cubre los [compromisos en los modelos de difusión](https://arxiv.org/abs/2206.00364)

### Resumen
Esta publicación está diseñada para mostrar la API central de `diffusers`, que se divide en tres componentes:
1. **Pipelines**: clases de alto nivel diseñadas para generar rápidamente muestras de modelos de difusión entrenados populares de manera fácil de usar.
2. **Modelos**: arquitecturas populares para entrenar nuevos modelos de difusión, *por ejemplo* [UNet](https://arxiv.org/abs/1505.04597).
3. **Schedulers** (planificadores): varias técnicas para generar imágenes a partir de ruido durante la *inferencia*, así como para generar imágenes ruidosas para el *entrenamiento*.

**Nota**: Este cuaderno se enfoca solo en la **inferencia**. Si desea obtener una guía más práctica sobre el **entrenamiento** de modelos de difusión, consulte el cuaderno [*Training with Diffusers*](https://colab.research.google.com/gist/anton-l/f3a8206dae4125b93f05b1f5f703191d/diffusers_training_example.ipynb).

### Instalar `diffusers`

In [ ]:
!pip install diffusers --upgrade

### Resumen

Uno de los objetivos de la librería `diffusers` es hacer que los modelos de difusión sean accesibles para una amplia gama de profesionales del *deep learning*.
Con esto en mente, nuestro objetivo fue construir una librería que sea **fácil de usar**, **intuitiva de entender** y **fácil de contribuir**.

Como un repaso rápido, los modelos de difusión son sistemas de *machine learning* que se entrenan para *eliminar el ruido* aleatorio gaussiano paso a paso, para llegar a una muestra de interés, como una *imagen*.

El modelo subyacente, a menudo una red neuronal, se entrena para predecir una forma de eliminar ligeramente el ruido de la imagen en cada paso. Después de un cierto número de pasos, se obtiene una muestra.

El proceso se ilustra con el siguiente diseño:
![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusion-process.png)

La arquitectura de la red neuronal, a la que nos referimos como **modelo**, comúnmente sigue la arquitectura UNet propuesta en [este artículo](https://arxiv.org/abs/1505.04597) y mejorada en el artículo Pixel++.

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/unet-model.png)

No se preocupe si no entiende todo. Algunos de los puntos destacados de la arquitectura son:
* este modelo predice imágenes del mismo tamaño que la entrada
* el modelo hace que la imagen de entrada pase por varios bloques de capas ResNet que reducen a la mitad el tamaño de la imagen en 2
* luego a través del mismo número de bloques que la vuelven a muestrear.
* las conexiones de salto (skip connections) enlazan características en la ruta de submuestreo con las capas correspondientes en la ruta de sobremuestreo.

El proceso de difusión consiste en tomar ruido aleatorio del tamaño de la salida deseada y pasarlo por el modelo varias veces. El proceso termina después de un número dado de pasos, y la imagen de salida debe representar una muestra de acuerdo con la distribución de datos de entrenamiento del modelo, por ejemplo, una imagen de una mariposa.

Durante el entrenamiento mostramos muchas muestras de una distribución dada, como imágenes de mariposas. Después del entrenamiento, el modelo podrá procesar ruido aleatorio para generar imágenes de mariposas similares.

Sin entrar en demasiados detalles, el modelo generalmente no se entrena para predecir directamente una imagen ligeramente menos ruidosa, sino para predecir el "residuo de ruido" que es la diferencia entre una imagen menos ruidosa y la imagen de entrada (para un modelo de difusión llamado "DDPM") o, de manera similar, el gradiente entre los dos pasos de tiempo (como el modelo de difusión llamado "Score VE").

Para el proceso de eliminación de ruido, es necesario un algoritmo de programación de ruido específico y "envolver" el modelo para definir cuántos pasos de difusión se necesitan para la inferencia, así como cómo *calcular* una imagen menos ruidosa a partir de la salida del modelo. Aquí es donde entran en juego los diferentes **planificadores** (schedulers) de la librería `diffusers`.

Finalmente, una **pipeline** agrupa un **modelo** y un **planificador** y facilita al usuario final la ejecución de un proceso completo de bucle de eliminación de ruido. Comenzaremos con las *pipelines* y profundizaremos en su implementación antes de examinar más de cerca los modelos y los planificadores.

## API Principal

### Pipelines

Comencemos importando una *pipeline*. Usaremos el modelo `google/ddpm-celebahq-256`, construido en colaboración por Google y U.C.Berkeley. Es un modelo que sigue el algoritmo [Denoising Diffusion Probabilistic Models (DDPM)](https://arxiv.org/abs/2006.11239) entrenado en un conjunto de datos de imágenes de celebridades.

Podemos importar la `DDPMPipeline`, lo que le permitirá realizar inferencias con un par de líneas de código:

In [ ]:
from diffusers import DDPMPipeline

El método `from_pretrained()` permite descargar el modelo y su configuración desde [el Hugging Face Hub](https://huggingface.co/google/ddpm-celebahq-256), un repositorio de más de 60,000 modelos compartidos por la comunidad.

In [ ]:
image_pipe = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256")
image_pipe.to("cuda")

Para generar una imagen, simplemente ejecutamos la *pipeline* y ni siquiera necesitamos darle ninguna entrada; generará una muestra de ruido inicial aleatoria y luego iterará el proceso de difusión.

La *pipeline* devuelve como salida un diccionario con una `muestra` generada de interés. Esto suele tardar entre 2 y 3 minutos en Google Colab:

In [ ]:
images = image_pipe().images

¡Echemos un vistazo! 🙂

In [ ]:
images[0]

¡Se ve bastante bien!

Ahora, intentemos entender un poco mejor lo que estaba pasando bajo el capó. Veamos de qué está hecha la *pipeline*:

In [ ]:
image_pipe

Podemos ver dentro de la *pipeline* un planificador y un modelo UNet. Echemos un vistazo más de cerca a ellos y a lo que esta *pipeline* hizo entre bastidores.

### Modelos

Las instancias de la clase de modelo son redes neuronales que toman una `muestra` ruidosa, así como un `paso de tiempo` como entradas para predecir una `muestra` de salida menos ruidosa. ¡Carguemos un modelo preentrenado y juguemos con él para entender la API del modelo!

Cargaremos un modelo de generación de imágenes incondicional simple de tipo `UNet2DModel` que fue lanzado con el [Artículo DDPM](https://arxiv.org/abs/2006.11239) y, por ejemplo, echemos un vistazo a otro punto de control entrenado en imágenes de iglesias: [`google/ddpm-church-256`](https://huggingface.co/google/ddpm-church-256).

De manera similar a lo que hemos visto para la clase *pipeline*, podemos cargar la configuración del modelo y los pesos con una línea, usando el método `from_pretrained()` con el que quizás esté familiarizado si ha jugado con la librería `transformers`:

In [ ]:
from diffusers import UNet2DModel

repo_id = "google/ddpm-church-256"
model = UNet2DModel.from_pretrained(repo_id, use_safetensors=False)

El método `from_pretrained()` almacena los pesos del modelo localmente, por lo que si ejecuta la celda anterior por segunda vez, será mucho más rápido. El modelo es una clase pura de PyTorch `torch.nn.Module` que puede ver al imprimir `model`.

In [ ]:
model

Ahora echemos un vistazo a la configuración del modelo. Se puede acceder a la configuración a través del atributo `config` y muestra todos los parámetros necesarios para definir la arquitectura del modelo (y solo esos).

In [ ]:
model.config

Como puede ver, la configuración del modelo es un diccionario congelado. Esto es para asegurar que la configuración **solo** se usará para definir la arquitectura del modelo en el momento de la instanciación y no para ningún atributo que pueda cambiarse durante la inferencia.

Un par de parámetros de configuración importantes son:
- `sample_size`: define las dimensiones de `altura` y `ancho` de la muestra de entrada.
- `in_channels`: define el número de canales de entrada de la muestra de entrada.
- `down_block_types` y `up_block_types`: definen el tipo de bloques de submuestreo y sobremuestreo que se utilizan para crear la arquitectura UNet, como se vio en la figura al principio de este cuaderno.
- `block_out_channels`: define el número de canales de salida de los bloques de submuestreo, también utilizados en orden inverso para el número de canales de entrada de los bloques de sobremuestreo.
- `layers_per_block`: define cuántos bloques ResNet están presentes en cada bloque UNet.

Sabiendo cómo se ve una configuración de UNet, puede intentar rápidamente instanciar la misma arquitectura de modelo con pesos aleatorios. Para hacerlo, pasemos la configuración como un diccionario desempaquetado a la clase `UNet2DModel`.

In [ ]:
model_random = UNet2DModel(**model.config)

Genial, lo anterior creó un modelo inicializado aleatoriamente con la misma configuración que el anterior.

Si desea guardar el modelo que acaba de crear, puede usar el método `save_pretrained()`, que guarda tanto los pesos del modelo como la configuración del modelo en la carpeta proporcionada.

In [ ]:
model_random.save_pretrained("my_model")

Echemos un vistazo a los archivos que se guardaron en `my_model`.

In [ ]:
!ls my_model

`diffusion_pytorch_model.bin` es un archivo binario de PyTorch que almacena los pesos del modelo y `config.json` almacena la configuración del modelo.

Si desea reutilizar el modelo, simplemente puede usar el método `from_pretrained()` nuevamente, ya que carga tanto los puntos de control locales como los presentes en el Hub.

In [ ]:
model_random = UNet2DModel.from_pretrained("my_model")

Volviendo al modelo realmente entrenado, veamos ahora cómo puede usar el modelo para la inferencia. Primero, necesita una muestra gaussiana aleatoria con la forma de una imagen (`batch_size` $	imes$ `in_channels` $	imes$ `sample_size` $	imes$ `sample_size`). Tenemos un eje `batch` porque un modelo puede recibir múltiples ruidos aleatorios, un eje `channel` porque cada uno consta de múltiples canales (como rojo-verde-azul), y finalmente `sample_size` corresponde a la altura y el ancho.

In [ ]:
import torch

torch.manual_seed(0)

noisy_sample = torch.randn(
    1, model.config.in_channels, model.config.sample_size, model.config.sample_size
)
noisy_sample.shape

¡Es hora de hacer la inferencia!

Puede pasar la muestra ruidosa junto con un `timestep` a través del modelo. El paso de tiempo es importante para indicar al modelo "cuán ruidosa" es la imagen de entrada (más ruidosa al principio del proceso, menos ruidosa al final), para que el modelo sepa si está más cerca del principio o del final del proceso de difusión.

Como se explicó en la introducción, el modelo predice la imagen ligeramente menos ruidosa, la diferencia entre la imagen ligeramente menos ruidosa y la imagen de entrada o incluso algo más. Es importante leer detenidamente la [tarjeta del modelo](https://huggingface.co/google/ddpm-church-256) para saber en qué ha sido entrenado el modelo. En este caso, el modelo predice el residuo de ruido (diferencia entre la imagen ligeramente menos ruidosa y la imagen de entrada).

In [ ]:
with torch.no_grad():
    noisy_residual = model(sample=noisy_sample, timestep=2).sample

El `noisy_residual` predicho tiene exactamente la misma forma que la entrada y lo usamos para calcular una imagen ligeramente menos ruidosa. Confirmemos que las formas de salida coinciden:

In [ ]:
noisy_residual.shape

Genial.

Ahora, para resumir, los **modelos**, como `UNet2DModel` (módulos PyTorch), son redes neuronales parametrizadas entrenadas para *predecir* una imagen o residuo ligeramente menos ruidoso. Se definen por su `.config` y se pueden cargar desde el Hub, así como guardar y cargar localmente. El siguiente paso es aprender a combinar este **modelo** con el **planificador** (scheduler) correcto para poder generar imágenes.

### Schedulers (Planificadores)

Los **Schedulers** (Planificadores) son algoritmos envueltos en una clase de Python. Definen el calendario de ruido que se utiliza para añadir ruido al modelo durante el entrenamiento, y también definen el algoritmo para *computar* la muestra ligeramente menos ruidosa dada la salida del modelo (aquí `noisy_residual`). Este cuaderno se centra únicamente en cómo utilizar las clases de *scheduler* para la inferencia. Puede consultar este cuaderno para ver cómo utilizar los *schedulers* para el entrenamiento.

Es importante recalcar aquí que, si bien los *modelos* tienen pesos entrenables, los *schedulers* suelen ser *sin parámetros* (en el sentido de que no tienen pesos entrenables) y simplemente definen el algoritmo para calcular la muestra ligeramente menos ruidosa. Por lo tanto, los *schedulers* no heredan de `torch.nn.Module`, pero al igual que los modelos se instancian mediante una configuración.

Para descargar la configuración de un *scheduler* del Hub, puede utilizar el método `from_config()` para cargar una configuración e instanciar un *scheduler*.

Usemos `DDPMScheduler`, el algoritmo de eliminación de ruido propuesto en el [Artículo DDPM](https://arxiv.org/abs/2006.11239).

In [ ]:
from diffusers import DDPMScheduler

scheduler = DDPMScheduler.from_pretrained(repo_id)

Echemos también un vistazo a la configuración aquí.

In [ ]:
scheduler.config

Los diferentes *schedulers* suelen definirse por diferentes parámetros. Para entender mejor para qué se utilizan exactamente los parámetros, se aconseja al lector que consulte directamente los archivos de *scheduler* correspondientes en `src/diffusers/schedulers/`, como el archivo [`src/diffusers/schedulers/scheduling_ddpm.py`](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_ddpm.py). Aquí están los más importantes:
- `num_train_timesteps` define la duración del proceso de eliminación de ruido, por ejemplo, cuántos pasos de tiempo se necesitan para procesar el ruido gaussiano aleatorio a una muestra de datos.
- `beta_schedule` define el tipo de calendario de ruido que se utilizará para la inferencia y el entrenamiento.
- `beta_start` y `beta_end` definen el valor de ruido más pequeño y el valor de ruido más alto del calendario.

Al igual que los *modelos*, los *schedulers* se pueden guardar y cargar con `save_config()` y `from_config()`.

In [ ]:
scheduler.save_config("my_scheduler")
new_scheduler = DDPMScheduler.from_pretrained("my_scheduler")

Todos los *schedulers* proporcionan uno o varios métodos `step()` que se pueden utilizar para calcular la imagen ligeramente menos ruidosa. El método `step()` puede variar de un *scheduler* a otro, pero normalmente espera al menos la salida del modelo, el `timestep` y la `noisy_sample` actual.

Tenga en cuenta que el método `step()` es una función de caja negra que "simplemente funciona". Si está interesado en comprender mejor cómo se calcula exactamente la muestra ruidosa anterior, tal como se define en el artículo original del *scheduler*, debe consultar el código real, *por ejemplo*, [haga clic aquí](https://github.com/huggingface/diffusers/blob/936cd08488260a9df3548d66628b83bc7f26bd9e/src/diffusers/schedulers/scheduling_ddpm.py#L130) para DDPM, que contiene comentarios y referencias al artículo original.

Probémoslo utilizando la salida del modelo de la sección anterior.

In [ ]:
less_noisy_sample = scheduler.step(
    model_output=noisy_residual, timestep=2, sample=noisy_sample
).prev_sample
less_noisy_sample.shape

Puede ver que la muestra calculada tiene exactamente la misma forma que la entrada del modelo, lo que significa que está listo para pasarla al modelo nuevamente en un siguiente paso.

Ahora, unamos todo y definamos el bucle de eliminación de ruido. Este bucle imprime las muestras (cada vez menos) ruidosas a lo largo del camino para una mejor visualización en el bucle de eliminación de ruido. Definamos una función de visualización que se encargue de posprocesar la imagen eliminada de ruido, convertirla a `PIL.Image` y mostrarla.

In [ ]:
import PIL.Image
import numpy as np

def display_sample(sample, i):
    image_processed = sample.cpu().permute(0, 2, 3, 1)
    image_processed = (image_processed + 1.0) * 127.5
    image_processed = image_processed.numpy().astype(np.uint8)

    image_pil = PIL.Image.fromarray(image_processed[0])
    display(f"Image at step {i}")
    display(image_pil)

Antes de definir el bucle, movamos la entrada y el modelo a la GPU para acelerar un poco el proceso de eliminación de ruido.

In [ ]:
model.to("cuda")
noisy_sample = noisy_sample.to("cuda")

¡Es hora de definir finalmente el bucle de eliminación de ruido! Es bastante sencillo para DDPM.

1. Predecir el residuo de la muestra menos ruidosa con el modelo.
2. Calcular la muestra menos ruidosa con el *scheduler*.

Además, cada 50 pasos mostrará el progreso.

Es importante destacar aquí que se itera sobre `scheduler.timesteps`, que es un tensor que define la secuencia de pasos de tiempo sobre los que iterar durante el proceso de eliminación de ruido. Por lo general, el proceso de eliminación de ruido va en orden decreciente de pasos de tiempo, es decir, desde el número total de pasos de tiempo (aquí 1000) hasta 0.

Dependiendo de su GPU, esto puede tardar hasta un minuto, tiempo suficiente para reflexionar sobre todo lo que ha aprendido hasta ahora mientras observa cómo se construye una iglesia de la nada, solo ruido ⛪.

In [ ]:
import tqdm

sample = noisy_sample

for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
  # 1. predict noise residual
  with torch.no_grad():
      residual = model(sample, t).sample

  # 2. compute less noisy image and set x_t -> x_t-1
  sample = scheduler.step(residual, t, sample).prev_sample

  # 3. optionally look at image
  if (i + 1) % 50 == 0:
      display_sample(sample, i + 1)

Puedes ver que lleva bastante tiempo ver una forma algo significativa, solo después de *aprox.* 800 pasos.

Aunque la calidad de la imagen es bastante buena, es posible que desee acelerar la generación de imágenes.

Para ello, puede intentar reemplazar el *scheduler* DDPM por el *scheduler* [DDIM](https://arxiv.org/abs/2010.02502), que mantiene una alta calidad de generación a un tiempo de generación significativamente acelerado.

**Intercambio de *schedulers***: una de las perspectivas emocionantes de una librería de modelos de difusión es que diferentes protocolos de programación pueden funcionar con diferentes modelos, ¡pero no existe una solución única para todos!
En este caso, DDIM funcionó como un sustituto de DDPM, pero esto no es universal (y representa un problema de investigación interesante).

Los *schedulers* DDPM y DDIM comparten más o menos la misma configuración, por lo que puede cargar un *scheduler* DDIM desde un *scheduler* DDPM.

In [ ]:
from diffusers import DDIMScheduler

scheduler = DDIMScheduler.from_config(repo_id)

El *scheduler* DDIM permite al usuario definir cuántos pasos de eliminación de ruido deben ejecutarse en la inferencia a través del método `set_timesteps`. El *scheduler* DDPM ejecuta por defecto 1000 pasos de eliminación de ruido. Reduzcamos significativamente este número a solo 50 pasos de inferencia para DDIM.

In [ ]:
scheduler.set_timesteps(num_inference_steps=50)

Y puedes ejecutar el mismo bucle que antes, solo que ahora estás utilizando el *scheduler* DDIM, mucho más rápido.

In [ ]:
import tqdm

sample = noisy_sample

for i, t in enumerate(tqdm.tqdm(scheduler.timesteps)):
  # 1. predict noise residual
  with torch.no_grad():
      residual = model(sample, t).sample

  # 2. compute previous image and set x_t -> x_t-1
  sample = scheduler.step(residual, t, sample).prev_sample

  # 3. optionally look at image
  if (i + 1) % 10 == 0:
      display_sample(sample, i + 1)

Puedes ver que la generación de imágenes es de hecho mucho más rápida, solo dos segundos, pero también que pagas sacrificando la calidad de la imagen a cambio de velocidad.

Genial, ahora debería tener una buena primera comprensión de los *schedulers*. Las cosas importantes a recordar son:
1. los *schedulers* son *sin parámetros* (sin pesos entrenables)
2. los *schedulers* definen el algoritmo que calcula la muestra ligeramente menos ruidosa durante la inferencia

Hay muchos *schedulers* ya añadidos a `diffusers` y `diffusers` añadirá aún más en el futuro. Es importante que lea las tarjetas de los modelos para entender qué puntos de control de modelo se pueden usar con qué *schedulers*.
Puede encontrar todos los *schedulers* disponibles [aquí](https://github.com/huggingface/diffusers/tree/main/src/diffusers/schedulers).

Para finalizar el capítulo sobre **modelos** y **schedulers** (planificadores), tenga en cuenta también que *deliberadamente* intentamos mantener los *modelos* y los *schedulers* lo más independientes posible. Esto significa que un `scheduler` nunca debe aceptar un `model` como entrada y viceversa. El modelo *predice* el residuo de ruido o la imagen ligeramente menos ruidosa con sus pesos entrenados, mientras que el *scheduler* *calcula* la muestra anterior dada la salida del modelo.

### Stable Diffusion

Ahora que tienes una comprensión de las *pipelines*, los modelos y los *schedulers*, veamos un ejemplo más complejo y popular: Stable Diffusion.

Stable Diffusion es un modelo de difusión latente de texto a imagen capaz de generar imágenes fotorrealistas a partir de cualquier entrada de texto. Es una herramienta poderosa que combina un codificador de texto, un modelo U-Net y un VAE (AutoCodificador Variacional) para lograr resultados impresionantes.

Antes de empezar, necesitamos instalar algunas librerías adicionales.

In [ ]:
!pip install transformers accelerate
!pip install invisible_watermark

Ahora, carguemos la *pipeline* de Stable Diffusion. Esta *pipeline* utiliza un punto de control de modelo pre-entrenado (por ejemplo, `runwayml/stable-diffusion-v1-5`) y maneja todo el proceso de generación de imágenes, desde la indicación de texto hasta la imagen.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_id = "runwayml/stable-diffusion-v1-5"
pipeline = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipeline.to("cuda")

Con la *pipeline* cargada, ahora podemos generar una imagen proporcionando una indicación de texto. Esto tomará un momento, ya que Stable Diffusion realiza muchos pasos para crear la imagen.

In [ ]:
prompt = "a photo of an astronaut riding a horse on mars"
image = pipeline(prompt).images[0]

¡Y ahí lo tienes! Una imagen única generada directamente a partir de tu indicación de texto usando Stable Diffusion.

In [ ]:
display(image)